# 02 · Nhúng embedding bằng BGE-M3

**Máy:** GPU (T4 hoặc P100) · **Thời gian:** ~20–40 phút cho toàn bộ · **Chi phí API:** 0 đồng

---

## Đọc kỹ trước khi chạy

> **Chạy bằng “Save Version” (Save & Run All), KHÔNG chạy tương tác.**
> Phiên tương tác tự ngắt khi để yên quá lâu — mất cả nửa tiếng nhúng.

**Quy trình:**

1. Settings → Accelerator → **GPU T4 x2** (hoặc P100)
2. Settings → Internet → **On** (để tải mô hình từ HuggingFace lần đầu)
3. Add Input → dataset `longmemeval` + output của notebook 01
4. Chạy thử với `SMOKE = True` trước (20 câu, ~2 phút) để kiểm tra đường ống
5. Đổi `SMOKE = False` → **Save Version** → đi làm việc khác

## Đầu ra

| File | Nội dung |
|---|---|
| `emb.npy` | Ma trận `float16` kích thước `[N, 1024]` |
| `texts.parquet` | Chỉ mục: `row_id` ↔ `question_id`, loại, cờ `has_answer`, nội dung |
| `manifest.json` | Tên mô hình, số chiều, ngày chạy — sáu tuần nữa nhìn lại sẽ cần |

Notebook 03 gắn output này làm input, **không nhúng lại** — nhờ vậy chỉnh biểu đồ
bao nhiêu lần cũng không tốn hạn mức GPU.

In [ ]:
!pip install -q FlagEmbedding 2>/dev/null || pip install -q sentence-transformers
print("xong")

In [ ]:
# ============ Cấu hình ============
from pathlib import Path
import json, gc, time
import numpy as np
import pandas as pd
import torch

DATA_DIR = Path("/kaggle/input/longmemeval")
NB01_DIR = Path("/kaggle/input/longmemeval-01-explore")   # sửa cho khớp tên output NB01
OUT_DIR  = Path("/kaggle/working")
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "BAAI/bge-m3"
MAX_LEN    = 512          # lượt hội thoại đều ngắn; 512 là thừa
BATCH      = 64           # giảm xuống 32 nếu hết VRAM
SMOKE      = True         # ĐỔI THÀNH False khi chạy thật
N_SMOKE    = 20

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ — bật Accelerator!")

In [ ]:
# ============ Gom toàn bộ văn bản cần nhúng ============
with open(DATA_DIR / "longmemeval_s.json", encoding="utf-8") as f:
    data = json.load(f)

if SMOKE:
    data = data[:N_SMOKE]
    print(f"CHẾ ĐỘ THỬ — chỉ {len(data)} instance\n")

rows = []
for ex in data:
    qid = ex["question_id"]
    ans_ids = set(ex.get("answer_session_ids") or [])

    rows.append({"question_id": qid, "kind": "question", "session_id": "",
                 "session_idx": -1, "turn_idx": -1, "role": "",
                 "has_answer": False, "text": ex["question"]})

    for si, (sid, sess) in enumerate(zip(ex["haystack_session_ids"],
                                         ex["haystack_sessions"])):
        for ti, t in enumerate(sess if isinstance(sess, list) else []):
            if not isinstance(t, dict):
                continue
            txt = (t.get("content") or "").strip()
            if not txt:
                continue
            rows.append({
                "question_id": qid, "kind": "turn", "session_id": sid,
                "session_idx": si, "turn_idx": ti, "role": t.get("role", ""),
                # cờ vàng: chỉ tính là bằng chứng khi phiên đó nằm trong answer_session_ids
                "has_answer": bool(t.get("has_answer")) and sid in ans_ids,
                "text": txt,
            })

texts = pd.DataFrame(rows).reset_index(drop=True)
texts["row_id"] = texts.index
print(f"{len(texts):,} đoạn cần nhúng "
      f"({(texts.kind=='question').sum()} câu hỏi + {(texts.kind=='turn').sum():,} lượt)")
print(f"Số lượt là bằng chứng: {texts.has_answer.sum():,}")
print(f"Độ dài ký tự — trung vị {texts.text.str.len().median():.0f}, "
      f"p95 {texts.text.str.len().quantile(0.95):.0f}")

In [ ]:
# ============ Nạp mô hình ============
t0 = time.time()
try:
    from FlagEmbedding import BGEM3FlagModel
    _m = BGEM3FlagModel(MODEL_NAME, use_fp16=True)
    def embed(batch_texts):
        return _m.encode(batch_texts, batch_size=BATCH,
                         max_length=MAX_LEN)["dense_vecs"]
    backend = "FlagEmbedding"
except Exception as e:
    print("FlagEmbedding không dùng được, chuyển sang sentence-transformers:", e)
    from sentence_transformers import SentenceTransformer
    _m = SentenceTransformer(MODEL_NAME, device="cuda")
    _m.max_seq_length = MAX_LEN
    def embed(batch_texts):
        return _m.encode(batch_texts, batch_size=BATCH,
                         normalize_embeddings=True, show_progress_bar=False)
    backend = "sentence-transformers"

print(f"Đã nạp mô hình bằng {backend} — {time.time()-t0:.0f}s")

In [ ]:
# ============ Nhúng theo lô, lưu từng chunk ============
CHUNK = 4096                                   # lưu tạm sau mỗi chunk
all_texts = texts["text"].tolist()
n = len(all_texts)
parts, t0 = [], time.time()

for start in range(0, n, CHUNK):
    part = embed(all_texts[start:start + CHUNK])
    parts.append(np.asarray(part, dtype=np.float16))
    done = min(start + CHUNK, n)
    el = time.time() - t0
    print(f"  {done:>7,}/{n:,}  ({done/n:5.1%})  "
          f"{el:6.0f}s trôi qua  ·  còn ~{el/done*(n-done):5.0f}s", flush=True)
    gc.collect(); torch.cuda.empty_cache()

emb = np.vstack(parts)
del parts; gc.collect()

# chuẩn hóa L2 để cosine = tích vô hướng (nhanh hơn nhiều ở notebook 03)
norms = np.linalg.norm(emb.astype(np.float32), axis=1, keepdims=True)
emb = (emb.astype(np.float32) / np.clip(norms, 1e-9, None)).astype(np.float16)

print(f"\nXong: {emb.shape}  ·  {emb.nbytes/1e6:.0f} MB  ·  {time.time()-t0:.0f}s")

In [ ]:
# ============ Lưu ============
np.save(OUT_DIR / "emb.npy", emb)
texts.drop(columns=[]).to_parquet(OUT_DIR / "texts.parquet", index=False)

manifest = {
    "model": MODEL_NAME,
    "backend": backend,
    "dim": int(emb.shape[1]),
    "n_rows": int(emb.shape[0]),
    "dtype": "float16",
    "l2_normalized": True,
    "max_length": MAX_LEN,
    "smoke": SMOKE,
    "n_instances": len(data),
    "created_at": pd.Timestamp.utcnow().isoformat(),
}
with open(OUT_DIR / "manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print(json.dumps(manifest, indent=2))
print("\nCÁC FILE ĐÃ LƯU:")
for p in sorted(OUT_DIR.glob("*")):
    print(f"  {p.name:22s} {p.stat().st_size/1e6:8.1f} MB")

In [ ]:
# ============ Kiểm tra nhanh: embedding có hợp lý không ============
# Với vài câu hỏi, cosine tới câu bằng chứng phải CAO HƠN cosine tới lượt ngẫu nhiên.
rng = np.random.default_rng(0)
qs = texts[texts.kind == "question"].head(5)

for _, q in qs.iterrows():
    qv = emb[q.row_id].astype(np.float32)
    same = texts[(texts.question_id == q.question_id) & (texts.kind == "turn")]
    pos = same[same.has_answer]
    neg = same[~same.has_answer]
    if not len(pos) or not len(neg):
        continue
    s_pos = float(np.mean(emb[pos.row_id.values].astype(np.float32) @ qv))
    s_neg = float(np.mean(emb[rng.choice(neg.row_id.values,
                                         min(50, len(neg)), replace=False)]
                          .astype(np.float32) @ qv))
    ok = "OK " if s_pos > s_neg else "NGỜ"
    print(f"  [{ok}] bằng chứng {s_pos:.3f}  vs  ngẫu nhiên {s_neg:.3f}   "
          f"{q.text[:60]}")

print("\nNếu dòng nào 'NGỜ' cũng xuất hiện thường xuyên thì kiểm tra lại "
      "cách gắn cờ has_answer trước khi chạy toàn bộ.")

## Bước tiếp theo

1. **Save Version** (Save & Run All) — chỉ những gì trong `/kaggle/working/` mới được giữ
2. Sang notebook 03 → **Add Input** → chọn **output của notebook này**
3. Notebook 03 chạy CPU, nạp `emb.npy` rồi tính Recall@k — **không nhúng lại**

Nếu sau này đổi mô hình embedding: đổi `MODEL_NAME`, chạy lại notebook này,
lưu thành version mới. `manifest.json` ghi lại mô hình nào tạo ra file nào,
nên không bao giờ trộn nhầm vector của hai mô hình.